In [ ]:
import numpy as np
import time

def cholesky_benchmark(n):
    """Time a Cholesky decomposition of an n×n PD matrix."""
    # Build a random positive definite matrix
    A = np.random.randn(n, n)
    K = A @ A.T + n * np.eye(n)  # Guaranteed PD

    start = time.time()
    L = np.linalg.cholesky(K)
    elapsed = time.time() - start
    return elapsed

ns = [10, 50, 100, 200, 500, 1000, 2000]
times = []
for n in ns:
    t = cholesky_benchmark(n)


    times.append(t)
    print(f"  n={n:5d}:  Cholesky time = {t:.4f} s")

print("\n=== SCALING ANALYSIS ===")
for i in range(1, len(ns)):
    ratio = times[i] / times[i-1]
    n_ratio = (ns[i] / ns[i-1])**3
    print(f"  n: {ns[i-1]:5d} → {ns[i]:5d}  "
          f"Time ratio: {ratio:.1f}x  "
          f"Expected (n³): {n_ratio:.1f}x")


In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import time

# === SAME DATA AS DAY 2 ===
x_obs = np.array([0.0, 1.0, 2.0, 4.0, 5.0])
y_obs = np.array([35.0, 28.0, 42.0, 31.0, 38.0])
X = x_obs[:, None]
y = y_obs

# === MODEL A: EXACT GP (Day 2 — for comparison) ===
print("=== EXACT GP (pm.gp.Marginal) ===")
t0 = time.time()

with pm.Model() as exact_model:
    ell = pm.InverseGamma("ell", alpha=3, beta=3)
    eta = pm.HalfCauchy("eta", beta=10)
    sigma = pm.HalfNormal("sigma", sigma=5)

    cov_func = eta**2 * pm.gp.cov.Matern52(input_dim=1, ls=ell)
    gp_exact = pm.gp.Marginal(cov_func=cov_func)
    y_ = gp_exact.marginal_likelihood("y_obs", X=X, y=y, sigma=sigma)

    trace_exact = pm.sample(
        draws=1000, tune=1000, chains=2,
        nuts_sampler="numpyro", target_accept=0.9,
        random_seed=42,
    )

t_exact = time.time() - t0
print(f"Exact GP sampling time: {t_exact:.1f} s")

# === MODEL B: HSGP APPROXIMATION ===
print("\n=== HSGP (pm.gp.HSGP) ===")
t0 = time.time()

with pm.Model() as hsgp_model:
    ell = pm.InverseGamma("ell", alpha=3, beta=3)
    eta = pm.HalfCauchy("eta", beta=10)
    sigma = pm.HalfNormal("sigma", sigma=5)

    # SAME kernel as before
    cov_func = eta**2 * pm.gp.cov.Matern52(input_dim=1, ls=ell)

    # HSGP instead of Marginal
    # m=[20]: 20 basis functions (plenty for 1D with 5 data points)
    # c=1.5:  extend domain by 50% beyond data range on each side
    gp_hsgp = pm.gp.HSGP(
        m=[20],       # Number of basis functions per dimension
        c=1.5,        # Boundary extension factor
        cov_func=cov_func,
    )

    # .prior() instead of .marginal_likelihood()
    # HSGP uses the latent GP formulation, not the marginal
    f = gp_hsgp.prior("f", X=X)

    # Likelihood: observed data = GP + noise
    y_ = pm.Normal("y_obs", mu=f, sigma=sigma, observed=y)

    trace_hsgp = pm.sample(
        draws=1000, tune=1000, chains=2,
        nuts_sampler="numpyro", target_accept=0.9,
        random_seed=42,
    )

t_hsgp = time.time() - t0
print(f"HSGP sampling time: {t_hsgp:.1f} s")
print(f"Speedup: {t_exact / t_hsgp:.1f}x")

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
var_names = ["ell", "eta", "sigma"]
var_labels = ["Length-scale ℓ (km)", "Amplitude η", "Noise σ"]

for ax, var, label in zip(axes, var_names, var_labels):
    # Exact GP posterior
    exact_samples = trace_exact.posterior[var].values.flatten()
    ax.hist(exact_samples, bins=40, density=True, alpha=0.5,
            color='red', label='Exact GP')

    # HSGP posterior
    hsgp_samples = trace_hsgp.posterior[var].values.flatten()
    ax.hist(hsgp_samples, bins=40, density=True, alpha=0.5,
            color='steelblue', label='HSGP')

    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Density')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    # Print numerical comparison
    print(f"{var}: Exact={exact_samples.mean():.3f}±{exact_samples.std():.3f}  "
          f"HSGP={hsgp_samples.mean():.3f}±{hsgp_samples.std():.3f}")

plt.suptitle('Hyperparameter Posteriors: Exact GP vs HSGP (m=20, c=1.5)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
X_new = np.linspace(-0.5, 5.5, 200)[:, None]

with hsgp_model:
    f_pred = gp_hsgp.conditional("f_pred", Xnew=X_new)
    ppc_hsgp = pm.sample_posterior_predictive(
        trace_hsgp, var_names=["f_pred"], random_seed=42
    )

f_hsgp = ppc_hsgp.posterior_predictive["f_pred"].values.reshape(-1, 200)
f_hsgp_mean = f_hsgp.mean(axis=0)
f_hsgp_q025 = np.percentile(f_hsgp, 2.5, axis=0)
f_hsgp_q975 = np.percentile(f_hsgp, 97.5, axis=0)
